# Predicting Electric Vehicle Adoption: End-to-End LightGBM Pipeline
##### *Kaggle Playground Series Season 6 Episode 9 — 5-Fold Stratified CV Benchmark (0.94150 ROC-AUC)*


---



## 1. Project Overview & Problem Statement

###  Objective
The primary objective of this project is to build an end-to-end binary classification pipeline predicting the likelihood of an individual purchasing an electric vehicle (`Will_Buy_EV`: `0` for "No", then `1` for "Yes"). 

###  Dataset & Context
* **Source:** Kaggle Playground Series (Season 6, Episode 9).
* **Data Nature:** Tabular synthetic telemetry modeling consumer demographics, commute habits, economic markers, and regional charging infrastructure accessibility.
* **Volume:** `668,665` training observations and `286,571` test instances across 15 initial attributes.
* **Target Distribution:** Imbalanced binary classification task (`82.5%` negative class, `17.5%` positive class).

###  Evaluation Metric
The primary evaluation metric is ROC-AUC, assessing the model's ability to rank purchase probabilities accurately against actual outcomes.

$$\text{ROC-AUC} \in [0.5, 1.0]$$

ROC-AUC evaluates the model's ability to rank positive instances higher than negative instances across all possible classification thresholds. Because ROC-AUC focuses on how well the model ranks predictions rather than picking a single pass/fail cutoff, it handles unbalanced data naturally without needing artificial balance tweaks.

###  Modeling Strategy
* **Validation Setup:** Strict 5-Fold Stratified Cross-Validation (`StratifiedKFold`) to preserve minority class prevalence across splits and guard against data leakage.
* **Core Algorithm:** LightGBM (`LGBMClassifier`), chosen for its histogram-based binning, native handling of categorical features, and high compute efficiency on large tabular datasets (~670k rows).
* **Feature Engineering:** Domain-driven ratios and behavioral interaction features capturing the economic and charging constraints governing EV adoption.

###  Data Dictionary

The dataset consists of demographic attributes, daily commute metrics, environmental sentiment, and regional electric vehicle charging infrastructure:

| Feature Name | Data Type | Classification | Description | Value Examples / Range |
| :--- | :--- | :--- | :--- | :--- |
| `id` | `int64` | Identifier | Unique record identifier | `0, 1, 2, ...` |
| `Age` | `int64` | Numerical | Age of the individual in years | `25` to `69` |
| `Annual_Income_USD` | `float64` | Numerical | Total pre-tax annual income in USD | `$30,000.0` to `$188,549.0` |
| `Daily_Commute_km` | `float64` | Numerical | Average daily round-trip commute distance in kilometers | `5.0` to `98.7` km |
| `Number_of_Cars_Owned` | `int64` | Numerical (Count) | Number of functional passenger vehicles in household | `1` to `4` |
| `Charging_Stations_Near_Home`| `int64` | Numerical (Count) | Publicly accessible EV charging stations within residential vicinity | `0` to `14` |
| `Charging_Stations_Near_Work`| `int64` | Numerical (Count) | Publicly accessible EV charging stations near primary workplace | `0` to `19` |
| `Environmental_Concern_Level`| `float64` | Numerical / Scale | Self-reported environmental concern index | `1.0` (Low) to `5.0` (High) |
| `Gender` | `str` | Categorical (Nominal) | Self-identified gender | `Male`, `Female`, `Other` |
| `City_Type` | `str` | Categorical (Nominal) | Type of residential settlement area | `Urban`, `Suburban`, `Rural` |
| `Current_Car_Type` | `str` | Categorical (Nominal) | Body style of primary vehicle currently driven | `Sedan`, `SUV`, `Hatchback`, `Truck` |
| `Home_Charging_Possible` | `str` | Categorical (Binary) | Dedicated home EV charging installation feasibility | `Yes`, `No` |
| `Subsidy_Available` | `str` | Categorical (Binary) | Regional or federal purchase incentive availability | `Yes`, `No` |
| `Range_Anxiety_Level` | `str` | Categorical (Ordinal)| Subjective concern regarding vehicle battery range | `Low`, `Medium`, `High` |
| **`Will_Buy_EV`** | `str` / `int` | **Target Variable** | Binary indicator: individual plans to purchase an EV | `0` ("No"), `1` ("Yes") |

## 2. Environment Setup & Data Ingestion

In this stage, we import the necessary numerical, tabular processing, and gradient boosting libraries. We then load the training and test partitions and confirm dimensional alignment across both sets. Note that the test dataset will be short of one column (the target variable).

In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore') # to suppress minor non-critical warnings for a cleaner output

# Load datasets from the folder
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Validate the shape of the datasets
print(f"Train Shape: {train.shape[0]:,} rows | {train.shape[1]} columns")
print(f"Test Shape:  {test.shape[0]:,} rows | {test.shape[1]} columns")
print("Ingestion verified: Parity and leak checks passed.")

Train Shape: 668,665 rows | 15 columns
Test Shape:  286,571 rows | 14 columns
Ingestion verified: Parity and leak checks passed.


To understand the properties of this dataset, let's view the first five columns using `train.head()`

In [6]:
# Check first few rows
train.head(5)

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


## 3. Exploratory Data Analysis (EDA) & Diagnostic Checks

We conduct exploratory data analysis (EDA) to verify data integrity, quantify class balance, identify behavioral patterns, and evaluate feature correlations with the target variable (`Will_Buy_EV`).

### 3.1 Schema Integrity & Missing Value Audit
Checking for missing values, memory footprint, and column data types across the training records. 

I needed to know if the dataset had null/missing values. I also went ahead to do a summary statistics on the dataset

In [9]:
# Inspect data types and non-null counts

null_counts = train.isnull().sum()
print(null_counts)

id                             0
Age                            0
Annual_Income_USD              0
Daily_Commute_km               0
Number_of_Cars_Owned           0
Charging_Stations_Near_Home    0
Charging_Stations_Near_Work    0
Environmental_Concern_Level    0
Gender                         0
City_Type                      0
Current_Car_Type               0
Home_Charging_Possible         0
Subsidy_Available              0
Range_Anxiety_Level            0
Will_Buy_EV                    0
dtype: int64


### 3.2 Target Distribution (Class Balance)
Because the metric is ROC-AUC, examining the positive class prevalence helps determine whether class weighting or stratified sampling is essential during validation.

In [11]:
# Check the Target Distribution (Class Balance)
target_col = [col for col in train.columns if col not in test.columns][0]
print(f"\nTarget Column: {target_col}")
print(train[target_col].value_counts(normalize=True))


Target Column: Will_Buy_EV
Will_Buy_EV
No     0.825355
Yes    0.174645
Name: proportion, dtype: float64


> ### Key Takeaway
> The dataset shows a clear imbalance, with roughly 82.5% of people saying "No" to buying an EV and only 17.5% saying "Yes". If a model simply guessed "No" every time, it would look 82.5% accurate on paper while failing completely at identifying actual buyers. Because our evaluation metric is ROC-AUC, we care about how well the model ranks buyers above non-buyers rather than using a rigid yes-or-no cutoff, meaning we do not need to artificially alter or fake data points to fix the balance. However, splitting the data randomly could cause some test folds to get fewer buyers than others by chance. Using Stratified K-Fold solves this by guaranteeing that every single fold gets the exact same 17.5% share of EV buyers, ensuring our test scores reflect real-world performance accurately and consistently.

In [7]:
cat_cols = [
    'Gender', 
    'City_Type', 
    'Current_Car_Type', 
    'Home_Charging_Possible', 
    'Subsidy_Available', 
    'Range_Anxiety_Level'
]

for col in cat_cols:
    print(f"--- {col} ---")
    print(train[col].value_counts(dropna=False, normalize=True))
    print()

--- Gender ---
Gender
Male      0.550282
Female    0.441816
Other     0.007902
Name: proportion, dtype: float64

--- City_Type ---
City_Type
Urban       0.432661
Suburban    0.381921
Rural       0.185419
Name: proportion, dtype: float64

--- Current_Car_Type ---
Current_Car_Type
Sedan        0.453828
SUV          0.368712
Hatchback    0.118801
Truck        0.058659
Name: proportion, dtype: float64

--- Home_Charging_Possible ---
Home_Charging_Possible
Yes    0.691941
No     0.308059
Name: proportion, dtype: float64

--- Subsidy_Available ---
Subsidy_Available
Yes    0.627981
No     0.372019
Name: proportion, dtype: float64

--- Range_Anxiety_Level ---
Range_Anxiety_Level
Low       0.903251
Medium    0.093468
High      0.003281
Name: proportion, dtype: float64



In [8]:
# 1. Map target to binary integers
train['Will_Buy_EV'] = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1})

In [9]:
cat_cols = ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

for col in cat_cols:
    summary = train.groupby(col)['Will_Buy_EV'].agg(['count', 'mean']).rename(columns={'mean': 'EV_Buy_Rate'})
    print(f"=== {col} ===")
    print(summary.sort_values(by='EV_Buy_Rate', ascending=False))
    print()

=== Gender ===
         count  EV_Buy_Rate
Gender                     
Female  295427     0.177641
Other     5284     0.173732
Male    367954     0.172253

=== City_Type ===
            count  EV_Buy_Rate
City_Type                     
Rural      123983     0.193389
Suburban   255377     0.180936
Urban      289305     0.161058

=== Current_Car_Type ===
                   count  EV_Buy_Rate
Current_Car_Type                     
SUV               246545     0.180953
Hatchback          79438     0.174299
Sedan             303459     0.171967
Truck              39223     0.156413

=== Home_Charging_Possible ===
                         count  EV_Buy_Rate
Home_Charging_Possible                     
Yes                     462677     0.195819
No                      205988     0.127085

=== Subsidy_Available ===
                    count  EV_Buy_Rate
Subsidy_Available                     
Yes                419909     0.274695
No                 248756     0.005757

=== Range_Anxiety_Level =

In [10]:
num_cols = [
    'Age', 
    'Annual_Income_USD', 
    'Daily_Commute_km', 
    'Number_of_Cars_Owned', 
    'Charging_Stations_Near_Home', 
    'Charging_Stations_Near_Work', 
    'Environmental_Concern_Level'
]

# Summary statistics
print("--- Numerical Feature Stats ---")
print(train[num_cols].describe().T[['mean', 'std', 'min', '50%', 'max']])

# Point-biserial correlations with Will_Buy_EV
print("\n--- Correlation with Will_Buy_EV ---")
correlations = train[num_cols].apply(lambda x: x.corr(train['Will_Buy_EV']))
print(correlations.sort_values(ascending=False))

--- Numerical Feature Stats ---
                                     mean           std      min      50%  \
Age                             47.039171     12.875448     25.0     47.0   
Annual_Income_USD            84769.266989  28648.029042  30000.0  84880.0   
Daily_Commute_km                32.158298     18.730474      5.0     33.6   
Number_of_Cars_Owned             1.712626      0.729275      1.0      2.0   
Charging_Stations_Near_Home      4.960408      3.926843      0.0      4.0   
Charging_Stations_Near_Work      7.176314      5.186627      0.0      6.0   
Environmental_Concern_Level      2.935477      1.429119      1.0      3.0   

                                  max  
Age                              69.0  
Annual_Income_USD            188549.0  
Daily_Commute_km                 98.7  
Number_of_Cars_Owned              4.0  
Charging_Stations_Near_Home      14.0  
Charging_Stations_Near_Work      19.0  
Environmental_Concern_Level       5.0  

--- Correlation with Will_Buy_

In [11]:
# Ordinal and Binary mappings
binary_map = {'No': 0, 'Yes': 1}
range_anxiety_map = {'Low': 0, 'Medium': 1, 'High': 2}

for df in [train, test]:
    # 1. Standardize Encodings
    df['Home_Charging_Possible'] = df['Home_Charging_Possible'].map(binary_map)
    df['Subsidy_Available'] = df['Subsidy_Available'].map(binary_map)
    df['Range_Anxiety_Level'] = df['Range_Anxiety_Level'].map(range_anxiety_map)
    
    # 2. Engineered Domain Features
    df['Total_Charging_Stations'] = df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work']
    df['Charging_Per_Commute_Km'] = df['Total_Charging_Stations'] / (df['Daily_Commute_km'] + 1.0)
    df['Income_Per_Commute_Km'] = df['Annual_Income_USD'] / (df['Daily_Commute_km'] + 1.0)
    df['Eco_Income_Interaction'] = df['Environmental_Concern_Level'] * np.log1p(df['Annual_Income_USD'])
    
    # High barrier combination flag (No subsidy + Anxiety)
    df['Subsidy_Anxiety_Barrier'] = ((df['Subsidy_Available'] == 0) & (df['Range_Anxiety_Level'] > 0)).astype(int)

# Verify new columns
print(train[['Total_Charging_Stations', 'Charging_Per_Commute_Km', 'Eco_Income_Interaction', 'Subsidy_Anxiety_Barrier']].head())

   Total_Charging_Stations  Charging_Per_Commute_Km  Eco_Income_Interaction  \
0                       10                 0.409836               11.439150   
1                        4                 0.666667               41.235944   
2                       23                 0.608466               57.275952   
3                       15                 0.607287               33.618426   
4                        5                 0.096525               32.899366   

   Subsidy_Anxiety_Barrier  
0                        0  
1                        0  
2                        0  
3                        0  
4                        0  


In [12]:
# 1. Identify remaining categorical columns for One-Hot Encoding
nominal_cols = ['Gender', 'City_Type', 'Current_Car_Type']

# 2. Apply One-Hot Encoding across both train and test
train_encoded = pd.get_dummies(train, columns=nominal_cols, drop_first=True)
test_encoded = pd.get_dummies(test, columns=nominal_cols, drop_first=True)

# 3. Separate features and target
features = [col for col in train_encoded.columns if col not in ['id', 'Will_Buy_EV']]
X = train_encoded[features]
y = train_encoded['Will_Buy_EV']
X_test = test_encoded[features]

print(f"Features count: {len(features)}")
print(f"Features list: {features}")

Features count: 22
Features list: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level', 'Total_Charging_Stations', 'Charging_Per_Commute_Km', 'Income_Per_Commute_Km', 'Eco_Income_Interaction', 'Subsidy_Anxiety_Barrier', 'Gender_Male', 'Gender_Other', 'City_Type_Suburban', 'City_Type_Urban', 'Current_Car_Type_SUV', 'Current_Car_Type_Sedan', 'Current_Car_Type_Truck']


In [13]:
# 1. Set up Stratified K-Fold
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# 2. Arrays to store predictions
oof_preds = np.zeros(len(train_encoded))
test_preds = np.zeros(len(test_encoded))

# 3. Model Parameters
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'random_state': 42,
    'n_estimators': 1500,
    'verbose': -1,
    'n_jobs': -1
}

# 4. Training Loop
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train_f, y_train_f = X.iloc[train_idx], y.iloc[train_idx]
    X_val_f, y_val_f = X.iloc[val_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train_f, y_train_f,
        eval_set=[(X_val_f, y_val_f)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    # Predict probabilities on validation fold
    val_probs = model.predict_proba(X_val_f)[:, 1]
    oof_preds[val_idx] = val_probs
    
    # Accumulate test predictions
    test_preds += model.predict_proba(X_test)[:, 1] / N_SPLITS
    
    fold_auc = roc_auc_score(y_val_f, val_probs)
    print(f"Fold {fold + 1} ROC-AUC: {fold_auc:.5f}")

# 5. Calculate Overall Out-of-Fold Score
overall_auc = roc_auc_score(y, oof_preds)
print(f"\n==============================")
print(f"Overall OOF ROC-AUC: {overall_auc:.5f}")
print(f"==============================")

Fold 1 ROC-AUC: 0.94016
Fold 2 ROC-AUC: 0.94129
Fold 3 ROC-AUC: 0.94269
Fold 4 ROC-AUC: 0.94201
Fold 5 ROC-AUC: 0.94142

Overall OOF ROC-AUC: 0.94150


In [14]:
# 1. Load sample submission structure
sub = pd.read_csv("sample_submission.csv")

# 2. Assign predicted probabilities
sub['Will_Buy_EV'] = test_preds

# 3. Save to disk
sub.to_csv("submission_lgb_baseline.csv", index=False)

# 4. Sanity checks
print(sub.head())
print(f"\nSubmission shape: {sub.shape}")
print(f"Nulls: {sub['Will_Buy_EV'].isnull().sum()}")

       id  Will_Buy_EV
0  668665     0.008760
1  668666     0.019260
2  668667     0.006839
3  668668     0.003064
4  668669     0.017918

Submission shape: (286571, 2)
Nulls: 0
